# Description
- Modelo concatenando saidas não-dense, com Conv2D em reshape

### Imports

In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.layers import Input, Conv1D, Conv2D, Flatten, GlobalAveragePooling1D, LSTM, Dense, Concatenate, MaxPooling1D, MaxPooling2D
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import BatchNormalization

2025-06-23 20:03:04.705146: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-23 20:03:05.567393: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2025-06-23 20:03:05.571070: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-23 20:03:21.556372: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
%run ../UtilsNew.ipynb

### Load and filter data

In [3]:
raw_anual_df = pd.read_csv("../data/minas_gerais_bh.csv", sep=";")
anual_df = pre_processing(raw_anual_df)

anual_df["hora"] = anual_df["data_hora"].dt.hour
anual_df = filter_between(anual_df, "hora", 7, 18)

janela_tempo = 11

### Functions

In [4]:
def select_and_normalize_minmax(anual_df, list_cols):
    df = anual_df[list_cols]
    
    scaler = MinMaxScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns)
    
    return df_scaled.fillna(df_scaled.mean())

def select_and_normalize_divided(anual_df, list_cols):

    def divided_by_1000(n):
        return n/1000
        
    df = anual_df[list_cols]
    
    df_scaled = df.apply(divided_by_1000)
    
    return df_scaled.fillna(df_scaled.mean())

def create_model(janela_tempo, n_features):
    # CNN
    input_clima = Input(shape=(janela_tempo, n_features, 1), name="input_clima")
    
    x_cnn = Conv2D(filters=64, kernel_size=(2, 3), activation='relu', padding='same')(input_clima)
    x_cnn = BatchNormalization()(x_cnn)
    
    x_cnn = Conv2D(filters=32, kernel_size=(2, 3), activation='relu', padding='same')(x_cnn)
    x_cnn = BatchNormalization()(x_cnn)
    
    x_cnn = MaxPooling2D(pool_size=(2, 2))(x_cnn)
    
    x_cnn = Conv2D(filters=16, kernel_size=(2, 2), activation='relu', padding='same')(x_cnn)
    x_cnn = BatchNormalization()(x_cnn)

    x_cnn = Flatten()(x_cnn)
    
    # LSTM
    input_rad = Input(shape=(janela_tempo, 1), name="input_radiacao")
    x_lstm = LSTM(64, return_sequences=True)(input_rad)
    x_lstm = LSTM(32)(x_lstm)
    
    # Join models
    x = Concatenate()([x_cnn, x_lstm])
    x = Dense(128, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dense(64, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dense(32, activation='relu')(x)
    output = Dense(1, name="saida", activation='linear')(x)
    
    model = Model(inputs=[input_clima, input_rad], outputs=output)
    model.compile(optimizer='adam', loss='mse')
    
    return model


def training_model(modelo, Xc_train, Xr_train, y_train, Xc_val, Xr_val, y_val):
    modelo.fit(
        x={"input_clima": Xc_train, "input_radiacao": Xr_train},
        y=y_train,
        validation_data=(
            {"input_clima": Xc_val, "input_radiacao": Xr_val},
            y_val
        ),
        epochs=100,
        batch_size=32,
        verbose=0
    )
    
def criar_janelas(df_input, df_target, janela=12, horizonte=1):
    X_cnn, X_lstm, y = [], [], []
    
    for i in range(len(df_input) - janela - horizonte + 1):
        #Dados para CNN — formato (n_features, janela, 1)
        dados_cnn = df_input.iloc[i:i+janela].values.T  # Transpõe para (n_features, janela)
        dados_cnn = dados_cnn.reshape(dados_cnn.shape[0], dados_cnn.shape[1], 1)  # (n_features, janela, 1)
        
        #Dados para LSTM — formato (janela, 1)
        dados_lstm = df_target.iloc[i:i+janela].values.reshape(-1, 1)
        
        #Target — valor no horizonte futuro
        target = df_target.iloc[i+janela+horizonte-1]
        
        #Append
        X_cnn.append(dados_cnn)
        X_lstm.append(dados_lstm)
        y.append(target)
    
    return np.array(X_cnn), np.array(X_lstm), np.array(y)

def plot_results(pred, real, period):
    set_plot_size(12, 6)
    
    pred = pred.reshape(pred.shape[0])
    
    df_pred = pd.DataFrame(data=pred, index=period, columns=["predicted"])
    df_real = pd.DataFrame(data=real, index=period, columns=["real"])
    
    sns.lineplot(df_pred, palette=["red"], ci=None)
    sns.lineplot(df_real, palette=["blue"], ci=None)

def get_feature_importance(X_train, Y_train, cols):

    time_steps = 12
    n_features = 16

    model = Sequential([
        Input(shape=(n_features, 1)),
        Conv1D(32, kernel_size=2, activation='relu'),
        Conv1D(16, kernel_size=2, activation='relu'),
        GlobalAveragePooling1D(),
        Dense(1, activation='linear')  # Para regressão
    ])
        
    model.compile(optimizer='adam', loss='mse')
    
    model.fit(X_train, Y_train, epochs=10, verbose=0)
    
    X_sample = X_train[0:1]
    
    attributions = integrated_gradients(model, X_sample, m_steps=100)

    
    d = {"feature": cols, "value": attributions}
    
    local_df = pd.DataFrame(data=d)
    plt.xticks(rotation=90)
    sns.barplot(local_df, x="feature", y="value")


def integrated_gradients(model,
                          input_tensor,
                          target_class_idx=None,
                          baseline=None,
                          m_steps=100):
    """
    Compute Integrated Gradients for a Keras/TensorFlow model.
    """
    if baseline is None:
        baseline = np.zeros_like(input_tensor).astype(np.float32)

    input_tensor = input_tensor.astype(np.float32)
    baseline = baseline.astype(np.float32)

    interpolated_inputs = [
        baseline + (float(alpha) / m_steps) * (input_tensor - baseline)
        for alpha in range(0, m_steps + 1)
    ]
    interpolated_inputs = tf.convert_to_tensor(np.concatenate(interpolated_inputs, axis=0))

    with tf.GradientTape() as tape:
        tape.watch(interpolated_inputs)
        preds = model(interpolated_inputs)

        if target_class_idx is not None:
            outputs = preds[:, target_class_idx]
        else:
            outputs = tf.reduce_sum(preds, axis=1)  # Regressão

    grads = tape.gradient(outputs, interpolated_inputs)

    avg_grads = tf.reduce_mean(grads[:-1], axis=0)  # Exclui último ponto

    integrated_grads = (input_tensor - baseline) * avg_grads.numpy()

    return integrated_grads.squeeze()

def get_metrics(test_pred, y_val):
    mse = mean_squared_error(test_pred.reshape(test_pred.shape[0]), y_val)
    rmse = math.sqrt(mse)
    mae = mean_absolute_error(test_pred.reshape(test_pred.shape[0]), y_val)
    r2 = r2_score(test_pred.reshape(test_pred.shape[0]), y_val)

    return {"mse": mse, "rmse": rmse, "mae": mae, "r2": r2}

# Test with best 3 columns

In [5]:
list_cols = ["pto_orvalho_min_c", "umi_min_perc", "pressao_min_hpa", "radiacao"]
df_norm = select_and_normalize_divided(anual_df, list_cols)

df_inupt = df_norm
df_target = df_norm[["radiacao"]]

In [6]:
X_clima, X_rad, y = criar_janelas(df_inupt, df_target, janela=janela_tempo, horizonte=1)

X_clima = X_clima.reshape(X_clima.shape[0], X_clima.shape[2], X_clima.shape[1], 1)

Xc_train, Xc_val, Xr_train, Xr_val, y_train, y_val = train_test_split(X_clima, X_rad, y, test_size=0.3, random_state=42, shuffle=False)

In [33]:
results = {}

def process(times):
    for i in range(times):
        print("Starting test " + str(i))
        results[i] = {"train_pred":None, "test_pred": None, "period": None, "metrics":None}

        model = create_model(janela_tempo, df_inupt.shape[1])
        training_model(model, Xc_train, Xr_train, y_train, Xc_val, Xr_val, y_val)
    
        train_pred = model.predict(x={"input_clima": Xc_train, "input_radiacao": Xr_train})
        test_pred = model.predict(x={"input_clima": Xc_val, "input_radiacao": Xr_val})
    
        period = anual_df[train_pred.shape[0]:]['data'].iloc[:test_pred.shape[0]]
        # plot_results(test_pred.reshape(test_pred.shape[0]), y_val, period)

        results[i]["train_pred"] = train_pred
        results[i]["test_pred"] = test_pred
        results[i]["metrics"] = get_metrics(test_pred, y_val)

process(15)

Starting test 0
42/42 [==============================] - 0s 6ms/step
Starting test 1
42/42 [==============================] - 0s 6ms/step
Starting test 2
42/42 [==============================] - 0s 8ms/step
Starting test 3
42/42 [==============================] - 0s 5ms/step
Starting test 4
42/42 [==============================] - 0s 6ms/step
Starting test 5
42/42 [==============================] - 0s 6ms/step
Starting test 6
42/42 [==============================] - 0s 6ms/step
Starting test 7
42/42 [==============================] - 0s 6ms/step
Starting test 8
42/42 [==============================] - 0s 5ms/step
Starting test 9
42/42 [==============================] - 0s 5ms/step
Starting test 10
42/42 [==============================] - 0s 6ms/step
Starting test 11
42/42 [==============================] - 0s 5ms/step
Starting test 12
42/42 [==============================] - 0s 5ms/step
Starting test 13
42/42 [==============================] - 0s 5ms/step
Starting test 14
42/42 [======

In [34]:
for i in results:
    print(results[i]["metrics"])
    print("\n")

{'mse': 0.31403801632673517, 'rmse': 0.5603909495403501, 'mae': 0.41284471892457497, 'r2': 0.6330029398873214}


{'mse': 0.2685211909732706, 'rmse': 0.5181903038201995, 'mae': 0.3728013976733585, 'r2': 0.7205667259033058}


{'mse': 0.2913080712512847, 'rmse': 0.5397296279168716, 'mae': 0.3833812913341939, 'r2': 0.6962612069992618}


{'mse': 0.24736048168212244, 'rmse': 0.4973534776013157, 'mae': 0.36137991925592206, 'r2': 0.7282646777533396}


{'mse': 0.2697468125611956, 'rmse': 0.519371555402484, 'mae': 0.380733144624333, 'r2': 0.7172154351441831}


{'mse': 0.2620179239812118, 'rmse': 0.5118768640808176, 'mae': 0.37292954516370935, 'r2': 0.690490004415871}


{'mse': 0.28201520630394206, 'rmse': 0.5310510392645345, 'mae': 0.3816235107231321, 'r2': 0.7087289308155827}


{'mse': 0.2854026669268128, 'rmse': 0.5342309116167023, 'mae': 0.3842921329381892, 'r2': 0.6860444807439194}


{'mse': 0.2834766463756135, 'rmse': 0.5324252495661842, 'mae': 0.38732558554758134, 'r2': 0.7164155579643889}